# InoveHub
## Sistema de Incubadora de Empresas

### CRUD Empresa, Investimentos, Contato e Evento com Gráficos

In [ ]:
import os
import pandas as pd
import panel as pn
import plotly.express as px
from sqlalchemy import create_engine, text
from datetime import date, datetime
from dotenv import load_dotenv

pn.extension('plotly', 'tabulator') 
load_dotenv()

DB_URL = os.getenv("DATABASE_URL")
if not DB_URL:
    raise ValueError("❌ ERRO: DATABASE_URL não encontrada no arquivo .env")

engine = create_engine(DB_URL)

def get_data(table_name):
    try:
        with engine.connect() as conn:
            return pd.read_sql(f"SELECT * FROM {table_name}", conn)
    except Exception as e:
        print(f"Aviso: Não foi possível ler a tabela {table_name}. Erro: {e}")
        return pd.DataFrame()

def execute_sql(sql_query, params=None):
    try:
        with engine.connect() as conn:
            conn.execute(text(sql_query), params or {})
            conn.commit()
        return True, "Sucesso!"
    except Exception as e:
        return False, str(e)

btn_atualizar_dash = pn.widgets.Button(name='🔄 Atualizar Dados', button_type='primary', width=150)

def carregar_dados_dash():
    df_e = get_data("Empresa")
    
    df_i = get_data("Investimento")
    if not df_i.empty and 'data_aporte' in df_i.columns:
        df_i['data_aporte'] = pd.to_datetime(df_i['data_aporte'])
        
    df_c = get_data("Contato")
    
    df_evt = get_data("Evento")
    if not df_evt.empty and 'data_hora_inicio' in df_evt.columns:
        df_evt['data_hora_inicio'] = pd.to_datetime(df_evt['data_hora_inicio'])

    df_p = get_data("Projeto")
    df_m = get_data("Mentor")
        
    return df_e, df_i, df_c, df_evt, df_p, df_m

_df_e, _, _, _, _, _ = carregar_dados_dash()
areas_disponiveis = list(_df_e['area_atuacao'].unique()) if not _df_e.empty else []

filtro_area = pn.widgets.MultiChoice(name='Filtrar Área (Empresas)', options=areas_disponiveis, value=areas_disponiveis, solid=False)
slider_top_cargos = pn.widgets.IntSlider(name='Top Cargos', start=3, end=20, value=10)

def criar_grafico_funil(areas, clique):
    df_e, _, _, _, _, _ = carregar_dados_dash()
    if areas: df_e = df_e[df_e['area_atuacao'].isin(areas)]
    if df_e.empty: return pn.pane.Markdown("### Sem dados de empresas")
    contagem = df_e['status_atual'].value_counts().reset_index()
    contagem.columns = ['Status', 'Quantidade']
    mapa_cores = {'Ativa': '#636EFA', 'Pendente': '#EF553B', 'Inativa': '#00CC96', 'Graduada': '#AB63FA'}
    return px.bar(contagem, x='Status', y='Quantidade', color='Status', title="Status das Empresas", template="plotly_white", color_discrete_map=mapa_cores)

def criar_grafico_invest(clique):
    _, df_i, _, _, _, _ = carregar_dados_dash()
    if df_i.empty: return pn.pane.Markdown("### Sem investimentos")
    df_agrupado = df_i.groupby(df_i['data_aporte'].dt.to_period('M').astype(str))['valor'].sum().reset_index()
    return px.line(df_agrupado, x='data_aporte', y='valor', markers=True, title="Evolução Financeira", template="plotly_white")

def criar_grafico_cargos(top_n, clique):
    _, _, df_c, _, _, _ = carregar_dados_dash()
    if df_c.empty: return pn.pane.Markdown("### Sem contatos")
    top_cargos = df_c['cargo'].value_counts().nlargest(top_n).reset_index()
    top_cargos.columns = ['Cargo', 'Quantidade']
    return px.bar(top_cargos, y='Cargo', x='Quantidade', orientation='h', title=f"Top {top_n} Cargos", template="plotly_white")

def criar_grafico_eventos(clique):
    _, _, _, df_evt, _, _ = carregar_dados_dash()
    if df_evt.empty: return pn.pane.Markdown("### Sem eventos registrados")
    df_agrupado = df_evt.groupby(df_evt['data_hora_inicio'].dt.to_period('M').astype(str)).size().reset_index(name='Quantidade')
    fig = px.line(df_agrupado, x='data_hora_inicio', y='Quantidade', markers=True, title="📅 Eventos Realizados por Mês", template="plotly_white")
    fig.update_traces(line_color='#FFA15A') 
    return fig

def criar_grafico_projetos(clique):
    _, _, _, _, df_p, _ = carregar_dados_dash()
    if df_p.empty: return pn.pane.Markdown("### Sem projetos")
    contagem = df_p['status_projeto'].value_counts().reset_index()
    contagem.columns = ['Status', 'Quantidade']
    return px.pie(contagem, values='Quantidade', names='Status', title="Projetos por Status", template="plotly_white")

def criar_grafico_mentores(clique):
    _, _, _, _, _, df_m = carregar_dados_dash()
    if df_m.empty: return pn.pane.Markdown("### Sem mentores")
    contagem = df_m['area_especialidade'].value_counts().reset_index()
    contagem.columns = ['Área', 'Quantidade']
    return px.bar(contagem, x='Quantidade', y='Área', orientation='h', title="Mentores por Especialidade", template="plotly_white")

view_funil = pn.bind(criar_grafico_funil, areas=filtro_area, clique=btn_atualizar_dash)
view_invest = pn.bind(criar_grafico_invest, clique=btn_atualizar_dash)
view_cargos = pn.bind(criar_grafico_cargos, top_n=slider_top_cargos, clique=btn_atualizar_dash)
view_eventos = pn.bind(criar_grafico_eventos, clique=btn_atualizar_dash)
view_projetos = pn.bind(criar_grafico_projetos, clique=btn_atualizar_dash)
view_mentores = pn.bind(criar_grafico_mentores, clique=btn_atualizar_dash)

layout_dash = pn.Column(
    pn.Row(btn_atualizar_dash),
    pn.Row(filtro_area, slider_top_cargos),
    pn.Row(view_funil, view_invest),
    pn.Row(view_projetos, view_mentores),
    pn.Row(view_eventos, view_cargos)
)

def criar_aba_empresa():
    # Widget de Busca
    i_busca = pn.widgets.TextInput(name='🔍 Buscar Empresa', placeholder='Filtrar por Nome ou CNPJ...')
    
    tabela = pn.widgets.Tabulator(pd.DataFrame(), pagination='remote', page_size=10, sizing_mode='stretch_width')
    i_cnpj = pn.widgets.TextInput(name='CNPJ', placeholder='00.000.000/0000-00')
    i_nome = pn.widgets.TextInput(name='Nome')
    i_area = pn.widgets.Select(name='Área', options=['FinTech', 'HealthTech', 'AgroTech', 'EdTech', 'Logística', 'Varejo'])
    i_status = pn.widgets.Select(name='Status', options=['Ativa', 'Pendente', 'Inativa', 'Graduada', 'Acelerada'])
    btn_save = pn.widgets.Button(name='Salvar / Atualizar', button_type='primary')
    btn_del = pn.widgets.Button(name='Excluir', button_type='danger')
    msg = pn.pane.Markdown("")

    def carregar(event=None):
        df = get_data("Empresa")
        # Filtro de Busca
        term = i_busca.value.strip().lower()
        if term and not df.empty:
            # Procura no Nome OU no CNPJ
            df = df[df['nome'].str.lower().str.contains(term, na=False) | 
                    df['cnpj'].str.contains(term, na=False)]
        tabela.value = df

    def salvar(e):
        sql = """INSERT INTO Empresa (cnpj, nome, area_atuacao, status_atual) VALUES (:cnpj, :nome, :area, :status)
                 ON CONFLICT (cnpj) DO UPDATE SET nome=EXCLUDED.nome, area_atuacao=EXCLUDED.area_atuacao, status_atual=EXCLUDED.status_atual;"""
        ok, m = execute_sql(sql, {'cnpj': i_cnpj.value, 'nome': i_nome.value, 'area': i_area.value, 'status': i_status.value})
        msg.object = f"✅ {m}" if ok else f"❌ {m}"
        carregar()
    def excluir(e):
        ok, m = execute_sql("DELETE FROM Empresa WHERE cnpj = :cnpj", {'cnpj': i_cnpj.value})
        msg.object = f"🗑️ {m}" if ok else f"❌ {m}"
        carregar()
    def ao_clicar(e):
        if e.row >= 0:
            row = tabela.value.iloc[e.row]
            i_cnpj.value, i_nome.value = str(row['cnpj']), str(row['nome'])
            i_area.value, i_status.value = str(row['area_atuacao']), str(row['status_atual'])

    # Watcher: Atualiza tabela ao digitar
    i_busca.param.watch(carregar, 'value')
    
    btn_save.on_click(salvar); btn_del.on_click(excluir); tabela.on_click(ao_clicar)
    carregar()
    return pn.Column(pn.pane.Markdown("### 🏢 Gestão de Empresas"), i_busca, pn.Row(i_cnpj, i_nome, i_area, i_status), pn.Row(btn_save, btn_del), msg, tabela)


def criar_aba_projetos():
    i_busca = pn.widgets.TextInput(name='🔍 Buscar Projeto', placeholder='Filtrar por Título, ID ou CNPJ...')
    
    tabela = pn.widgets.Tabulator(pd.DataFrame(), pagination='remote', page_size=10, sizing_mode='stretch_width')
    i_titulo = pn.widgets.TextInput(name='Título')
    i_status = pn.widgets.Select(name='Status', options=['Em andamento', 'Concluído', 'Atrasado', 'Planejamento'])
    i_desc = pn.widgets.TextAreaInput(name='Descrição', height=100)
    i_prev = pn.widgets.DatePicker(name='Previsão Conclusão', value=date.today())
    i_cnpj = pn.widgets.Select(name='Empresa')
    i_id = pn.widgets.TextInput(name='ID', disabled=True, placeholder='Novo')
    btn_save = pn.widgets.Button(name='Salvar Projeto', button_type='success')
    btn_del = pn.widgets.Button(name='Excluir Projeto', button_type='danger')
    msg = pn.pane.Markdown("")

    def carregar(event=None):
        df = get_data("Projeto")
        term = i_busca.value.strip().lower()
        if term and not df.empty:
            # Procura em Título, ID ou CNPJ
            df = df[df['titulo'].str.lower().str.contains(term, na=False) |
                    df['projeto_id'].astype(str).str.contains(term, na=False) |
                    df['cnpj'].str.contains(term, na=False)]
        tabela.value = df
        
        df_emp = get_data("Empresa")
        i_cnpj.options = df_emp['cnpj'].tolist() if not df_emp.empty else []

    def salvar(e):
        sql = "INSERT INTO Projeto (titulo, status_projeto, descricao, data_previsao_conclusao, cnpj) VALUES (:tit, :st, :desc, :prev, :cnpj)"
        ok, m = execute_sql(sql, {'tit': i_titulo.value, 'st': i_status.value, 'desc': i_desc.value, 'prev': i_prev.value, 'cnpj': i_cnpj.value})
        msg.object = f"✅ Salvo!" if ok else f"❌ {m}"; carregar()
    def excluir(e):
        if not i_id.value or i_id.value == 'Novo': return
        ok, m = execute_sql("DELETE FROM Projeto WHERE projeto_id = :id", {'id': i_id.value})
        msg.object = f"🗑️ Deletado!" if ok else f"❌ {m}"; carregar()
    def ao_clicar(e):
        if e.row >= 0:
            row = tabela.value.iloc[e.row]; i_id.value = str(row['projeto_id']); i_titulo.value = str(row['titulo'])
            i_status.value = str(row['status_projeto']); i_cnpj.value = str(row['cnpj'])

    i_busca.param.watch(carregar, 'value')
    btn_save.on_click(salvar); btn_del.on_click(excluir); tabela.on_click(ao_clicar); carregar()
    return pn.Column(pn.pane.Markdown("### 🚀 Gestão de Projetos e Metas"), i_busca, pn.Row(i_titulo, i_status, i_prev), pn.Row(i_cnpj, i_id), i_desc, pn.Row(btn_save, btn_del), msg, tabela)

def criar_aba_mentores():
    i_busca = pn.widgets.TextInput(name='🔍 Buscar Mentor', placeholder='Filtrar por Nome ou Especialidade...')
    
    tabela = pn.widgets.Tabulator(pd.DataFrame(), pagination='remote', page_size=10, sizing_mode='stretch_width')
    i_nome = pn.widgets.TextInput(name='Nome')
    i_email = pn.widgets.TextInput(name='Email')
    i_tel = pn.widgets.TextInput(name='Telefone')
    i_area = pn.widgets.TextInput(name='Especialidade')
    i_bio = pn.widgets.TextAreaInput(name='Biografia', height=80)
    i_id = pn.widgets.TextInput(name='ID', disabled=True, placeholder='Novo')
    btn_save = pn.widgets.Button(name='Salvar Mentor', button_type='success')
    btn_del = pn.widgets.Button(name='Excluir Mentor', button_type='danger')
    msg = pn.pane.Markdown("")

    def carregar(event=None):
        df = get_data("Mentor")
        term = i_busca.value.strip().lower()
        if term and not df.empty:
            df = df[df['nome'].str.lower().str.contains(term, na=False) |
                    df['area_especialidade'].str.lower().str.contains(term, na=False)]
        tabela.value = df

    def salvar(e):
        sql = "INSERT INTO Mentor (nome, email, telefone, area_especialidade, biografia) VALUES (:nome, :email, :tel, :area, :bio)"
        ok, m = execute_sql(sql, {'nome': i_nome.value, 'email': i_email.value, 'tel': i_tel.value, 'area': i_area.value, 'bio': i_bio.value})
        msg.object = f"✅ Salvo!" if ok else f"❌ {m}"; carregar()
    def excluir(e):
        if not i_id.value or i_id.value == 'Novo': return
        ok, m = execute_sql("DELETE FROM Mentor WHERE mentor_id = :id", {'id': i_id.value})
        msg.object = f"🗑️ Deletado!" if ok else f"❌ {m}"; carregar()
    def ao_clicar(e):
        if e.row >= 0:
            row = tabela.value.iloc[e.row]; i_id.value = str(row['mentor_id']); i_nome.value = str(row['nome'])
            i_email.value = str(row['email']); i_area.value = str(row['area_especialidade'])

    i_busca.param.watch(carregar, 'value')
    btn_save.on_click(salvar); btn_del.on_click(excluir); tabela.on_click(ao_clicar); carregar()
    return pn.Column(pn.pane.Markdown("### 🧠 Gestão de Capital Humano (Mentores)"), i_busca, pn.Row(i_nome, i_email, i_tel), pn.Row(i_area, i_id), i_bio, pn.Row(btn_save, btn_del), msg, tabela)


def criar_aba_investimento():
    i_busca = pn.widgets.TextInput(name='🔍 Buscar Investimento', placeholder='Filtrar por Fonte ou CNPJ...')

    tabela = pn.widgets.Tabulator(pd.DataFrame(), pagination='remote', page_size=10, sizing_mode='stretch_width')
    i_fonte = pn.widgets.TextInput(name='Fonte'); i_forma = pn.widgets.Select(name='Forma', options=['Anjo', 'VC', 'Edital', 'Próprio'])
    i_data = pn.widgets.DatePicker(name='Data', value=date.today()); i_valor = pn.widgets.FloatInput(name='Valor', value=0.0, step=1000)
    i_cnpj = pn.widgets.Select(name='Empresa'); i_id = pn.widgets.TextInput(name='ID', disabled=True, placeholder='Novo')
    i_moeda = pn.widgets.Select(name='Moeda', options=['BRL', 'USD'])
    btn_save = pn.widgets.Button(name='Salvar', button_type='success'); btn_del = pn.widgets.Button(name='Excluir', button_type='danger'); msg = pn.pane.Markdown("")

    def carregar(event=None):
        df = get_data("Investimento")
        term = i_busca.value.strip().lower()
        if term and not df.empty:
            df = df[df['fonte_investidor'].str.lower().str.contains(term, na=False) |
                    df['cnpj'].str.contains(term, na=False)]
        tabela.value = df
        
        df_e = get_data("Empresa")
        i_cnpj.options = df_e['cnpj'].tolist() if not df_e.empty else []

    def salvar(e):
        sql = "INSERT INTO Investimento (fonte_investidor, forma_aporte, data_aporte, moeda, valor, cnpj) VALUES (:fonte, :forma, :data, :moeda, :valor, :cnpj)"
        ok, m = execute_sql(sql, {'fonte': i_fonte.value, 'forma': i_forma.value, 'data': i_data.value, 'moeda': i_moeda.value, 'valor': i_valor.value, 'cnpj': i_cnpj.value})
        msg.object = f"✅ Salvo!" if ok else f"❌ {m}"; carregar()
    def excluir(e):
        if not i_id.value or i_id.value == 'Novo': return
        ok, m = execute_sql("DELETE FROM Investimento WHERE investimento_id = :id", {'id': i_id.value})
        msg.object = f"🗑️ Deletado!" if ok else f"❌ {m}"; carregar()
    def ao_clicar(e):
        if e.row >= 0:
            row = tabela.value.iloc[e.row]; i_id.value = str(row['investimento_id']); i_fonte.value = str(row['fonte_investidor'])
            i_valor.value = float(row['valor']); i_cnpj.value = str(row['cnpj'])
    
    i_busca.param.watch(carregar, 'value')
    btn_save.on_click(salvar); btn_del.on_click(excluir); tabela.on_click(ao_clicar); carregar()
    return pn.Column(pn.pane.Markdown("### 💰 Gestão de Investimentos"), i_busca, pn.Row(i_fonte, i_forma, i_data), pn.Row(i_valor, i_moeda, i_cnpj, i_id), pn.Row(btn_save, btn_del), msg, tabela)


def criar_aba_contato():
    i_busca = pn.widgets.TextInput(name='🔍 Buscar Contato', placeholder='Filtrar por Nome, Email ou Empresa...')
    
    tabela = pn.widgets.Tabulator(pd.DataFrame(), pagination='remote', page_size=10, sizing_mode='stretch_width')
    i_nome = pn.widgets.TextInput(name='Nome'); i_cargo = pn.widgets.TextInput(name='Cargo')
    i_email = pn.widgets.TextInput(name='Email'); i_tel = pn.widgets.TextInput(name='Telefone')
    i_cnpj = pn.widgets.Select(name='Empresa'); i_id = pn.widgets.TextInput(name='ID', disabled=True, placeholder='Novo')
    btn_save = pn.widgets.Button(name='Salvar', button_type='success'); btn_del = pn.widgets.Button(name='Excluir', button_type='danger'); msg = pn.pane.Markdown("")

    def carregar(event=None):
        df = get_data("Contato")
        term = i_busca.value.strip().lower()
        if term and not df.empty:
            df = df[df['nome'].str.lower().str.contains(term, na=False) |
                    df['email'].str.lower().str.contains(term, na=False) |
                    df['cnpj'].str.contains(term, na=False)]
        tabela.value = df
        
        df_e = get_data("Empresa")
        i_cnpj.options = df_e['cnpj'].tolist() if not df_e.empty else []

    def salvar(e):
        sql = "INSERT INTO Contato (nome, cargo, email, telefone, cnpj) VALUES (:nome, :cargo, :email, :tel, :cnpj)"
        ok, m = execute_sql(sql, {'nome': i_nome.value, 'cargo': i_cargo.value, 'email': i_email.value, 'tel': i_tel.value, 'cnpj': i_cnpj.value})
        msg.object = f"✅ Salvo!" if ok else f"❌ {m}"; carregar()
    def excluir(e):
        if not i_id.value or i_id.value == 'Novo': return
        ok, m = execute_sql("DELETE FROM Contato WHERE contato_id = :id", {'id': i_id.value})
        msg.object = f"🗑️ Deletado!" if ok else f"❌ {m}"; carregar()
    def ao_clicar(e):
        if e.row >= 0:
            row = tabela.value.iloc[e.row]; i_id.value = str(row['contato_id']); i_nome.value = str(row['nome']); i_cargo.value = str(row['cargo']); i_cnpj.value = str(row['cnpj'])

    i_busca.param.watch(carregar, 'value')
    btn_save.on_click(salvar); btn_del.on_click(excluir); tabela.on_click(ao_clicar); carregar()
    return pn.Column(pn.pane.Markdown("### 👥 Gestão de Contatos"), i_busca, pn.Row(i_nome, i_cargo, i_email), pn.Row(i_tel, i_cnpj, i_id), pn.Row(btn_save, btn_del), msg, tabela)


def criar_aba_eventos():
    i_busca = pn.widgets.TextInput(name='🔍 Buscar Evento', placeholder='Filtrar por Título ou ID...')
    
    tabela = pn.widgets.Tabulator(pd.DataFrame(), pagination='remote', page_size=10, sizing_mode='stretch_width')
    i_titulo = pn.widgets.TextInput(name='Título do Evento')
    i_desc = pn.widgets.TextAreaInput(name='Descrição', height=100)
    i_inicio = pn.widgets.DatetimePicker(name='Início', value=datetime.now())
    i_fim = pn.widgets.DatetimePicker(name='Fim', value=datetime.now())
    i_id = pn.widgets.TextInput(name='ID', disabled=True, placeholder='Novo')
    btn_save = pn.widgets.Button(name='Agendar Evento', button_type='success')
    btn_del = pn.widgets.Button(name='Cancelar Evento', button_type='danger')
    msg = pn.pane.Markdown("")
    
    def carregar(event=None):
        df = get_data("Evento")
        term = i_busca.value.strip().lower()
        if term and not df.empty:
            df = df[df['titulo'].str.lower().str.contains(term, na=False) |
                    df['evento_id'].astype(str).str.contains(term, na=False)]
        tabela.value = df
        
    def salvar(e):
        sql = "INSERT INTO Evento (titulo, descricao, data_hora_inicio, data_hora_fim) VALUES (:tit, :desc, :ini, :fim)"
        ok, m = execute_sql(sql, {'tit': i_titulo.value, 'desc': i_desc.value, 'ini': i_inicio.value, 'fim': i_fim.value})
        msg.object = f"✅ Evento Agendado!" if ok else f"❌ {m}"; carregar()
    def excluir(e):
        if not i_id.value or i_id.value == 'Novo': return
        ok, m = execute_sql("DELETE FROM Evento WHERE evento_id = :id", {'id': i_id.value})
        msg.object = f"🗑️ Evento Cancelado!" if ok else f"❌ {m}"; carregar()
    def ao_clicar(e):
        if e.row >= 0:
            row = tabela.value.iloc[e.row]; i_id.value = str(row['evento_id']); i_titulo.value = str(row['titulo']); i_desc.value = str(row['descricao'])
            
    i_busca.param.watch(carregar, 'value')
    btn_save.on_click(salvar); btn_del.on_click(excluir); tabela.on_click(ao_clicar); carregar()
    
    return pn.Column(pn.pane.Markdown("### 📅 Gestão de Eventos & Agenda"), i_busca, pn.Row(i_titulo, i_inicio, i_fim), pn.Row(i_desc, i_id), pn.Row(btn_save, btn_del), msg, tabela)


abas = pn.Tabs(
    ("📊 Dashboard", layout_dash),
    ("🏢 Empresas", criar_aba_empresa()),
    ("🚀 Projetos", criar_aba_projetos()),
    ("🧠 Mentores", criar_aba_mentores()),
    ("💰 Investimentos", criar_aba_investimento()),
    ("👥 Contatos", criar_aba_contato()),
    ("📅 Eventos", criar_aba_eventos())
)

template = pn.template.FastListTemplate(
    title='InoveHub Manager',
    main=[abas],
    accent_base_color="#2ecc71",
    header_background="#2c3e50"
)

template.servable();
template.show()

Launching server at http://localhost:60080
